In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# 1. LOAD ALL 4 RAW CHANNELS
file_vis  = '/content/MSU-MR-1.png'                                          # Band 1: 0.6 µm Visible (Raw)
file_nir  = '/content/MSU-MR-2.png'                                          # Band 2: 0.8 µm NIR (Raw)
# FIX: The path must actually point to the uncorrected file, without the '_corrected' suffix.
file_swir = '/content/msu_mr_3.9_um_Shortwave_IR_(Uncalibrated).png'         # Band 3: 3.9 µm SWIR (Raw)
file_tir  = '/content/MSU-MR-4.png'                                          # Band 4: 10.8 µm TIR (Raw)

vis  = cv2.imread(file_vis, cv2.IMREAD_GRAYSCALE)
nir  = cv2.imread(file_nir, cv2.IMREAD_GRAYSCALE)
swir = cv2.imread(file_swir, cv2.IMREAD_GRAYSCALE)
tir  = cv2.imread(file_tir, cv2.IMREAD_GRAYSCALE)

if any(img is None for img in [vis, nir, swir, tir]):
    raise FileNotFoundError("One or more images failed to load. Check filenames and ensure .png format.")

# 2. SPATIAL ALIGNMENT (Minimum Bounding Crop)
# We crop all images to the smallest common dimension to guarantee perfectly matching arrays without warping.
min_h = min(vis.shape[0], nir.shape[0], swir.shape[0], tir.shape[0])
min_w = min(vis.shape[1], nir.shape[1], swir.shape[1], tir.shape[1])

vis  = vis[:min_h, :min_w]
nir  = nir[:min_h, :min_w]
swir = swir[:min_h, :min_w]
tir  = tir[:min_h, :min_w]

# 3. PRE-PROCESSING: SENSOR NOISE REDUCTION
vis = cv2.medianBlur(vis, 3)
nir = cv2.medianBlur(nir, 3)
swir = cv2.medianBlur(swir, 3)
tir = cv2.medianBlur(tir, 3)

# Normalize raw 8-bit digital numbers to float32 range [0.0, 1.0]
vis_f  = vis.astype(np.float32) / 255.0
nir_f  = nir.astype(np.float32) / 255.0
swir_f = swir.astype(np.float32) / 255.0
tir_f  = tir.astype(np.float32) / 255.0

# Mask out telemetry dropouts (Using VIS and NIR where lost packets are reliably black)
dropout_mask = (vis < 5) & (nir < 5)

# 4. SPECTRAL INDICES
eps = 1e-8
# NDVI: Distinguishes Water (negative) and Vegetation (positive)
ndvi = (nir_f - vis_f) / (nir_f + vis_f + eps)

# NDSI: Distinguishes Snow (high SWIR absorption) from Clouds (high SWIR reflection)
ndsi = (vis_f - swir_f) / (vis_f + swir_f + eps)

# 4. SPECTRAL INDICES & 4-BAND CLASSIFICATION
WATER      = 0
VEGETATION = 1
DESERT     = 2
CLOUD      = 3
SNOW_ICE   = 4
IGNORE     = 255

# Initialize the label array with the default class (Desert/Bare Soil)
labels = np.full(vis.shape, DESERT, dtype=np.uint8)

# Normalized Difference Vegetation Index (NDVI)
eps = 1e-8
ndvi = (nir_f - vis_f) / (nir_f + vis_f + eps)

# A. WATER: Revert to the highly reliable NDVI threshold.
# Water absorbs NIR heavily compared to VIS, yielding negative NDVI.
water_mask = (ndvi < -0.0)
labels[water_mask] = WATER

# B. VEGETATION: Strong NIR reflection, visible absorption, positive NDVI
veg_mask = (ndvi > 0.12) & (nir_f > 0.22)
labels[veg_mask] = VEGETATION

# C. CLOUDS: 4-Band Fusion Rule
# High Visible reflectance, high SWIR reflectance, low TIR (cold tops)
cloud_mask = (vis_f > 0.40) & (swir_f > 0.35) & (tir_f < 0.45)
labels[cloud_mask] = CLOUD

# D. SNOW / ICE: Bright in VIS, Absorbs SWIR, and Extremely Cold TIR
snow_mask = (vis_f > 0.50) & (ndsi > 0.30) & (tir_f < 0.30) & ~water_mask
labels[snow_mask] = SNOW_ICE

# D. TELEMETRY DROPOUTS
labels[dropout_mask] = IGNORE

# 6. VISUALIZATION & EXPORT
colors = ['#003366', '#2E7D32', '#D2B48C', '#FFFFFF', '#00FFFF']
cmap = ListedColormap(colors)

# Mask the IGNORE class to render as the black background
masked_labels = np.ma.masked_where(labels == IGNORE, labels)

# Set up a single figure for the 5-Class Fused Mask
fig, ax = plt.subplots(figsize=(12, 10))

# Plot the final 5-Class Fused Mask
seg_plot = ax.imshow(masked_labels, cmap=cmap, vmin=0, vmax=4)
ax.set_title("Fused Spectral Band Classification Mask", fontsize=14)
ax.axis('off')
ax.set_facecolor('black')

# Format the Colorbar
cbar = plt.colorbar(seg_plot, ax=ax, ticks=[0, 1, 2, 3, 4], shrink=0.8, pad=0.03)
cbar.ax.set_yticklabels(['Water', 'Vegetation', 'Desert/Soil', 'Cloud', 'Snow/Ice'], fontsize=11)

plt.tight_layout()
plt.savefig('/content/meteor_5class_mask.png', dpi=300, bbox_inches='tight')
plt.show()

# Export clean segmentation mask for Deep Learning pipelines
# cv2.imwrite('/content/meteor_5class_mask.png', labels)